# Subclassing `Layer` and `Model`

Following https://www.tensorflow.org/guide/keras/making_new_layers_and_models_via_subclassing.

## (1): Layer Class

In [49]:
import tensorflow as tf
from tensorflow import keras

Layers have (internal) "state": weights and biases. They also need a method to compute from an input to an output; that's the `call()` method.

In [50]:
class ZeroLinearOperator(keras.layers.Layer):
    
    def __init__(self, input_dimension = 5, number_of_nodes_next_layer = 5):

        print(f"Layer has {input_dimension*number_of_nodes_next_layer} total weights in a weight kernel of dimension {input_dimension}x{number_of_nodes_next_layer}")

        super().__init__() # inherit `Layer` properties and methods

        # adds weights to the layer:
        # [NOTE]: this MUST be `self.w` and NOT something like `self.weights`... HATE that!!
        self.w = self.add_weight(
            shape = (input_dimension, number_of_nodes_next_layer),
            initializer = 'zeros',
            trainable = True
        )

        # adds biases to the layer:
        # [NOTE]: SIMILARLY, this MUST be `self.b` and NOT `self.biases`...
        self.b = self.add_weight(
            shape = (number_of_nodes_next_layer, ), 
            initializer = "zeros",
            trainable = True
        )

    def call(self, inputs):
        # [NOTE]: this is a.b, which is NOT b.a; this operation is not commutitive
        return tf.matmul(inputs, self.w) + self.b

Since everything has been initialized to $0$, we expect that ANY input to the network just result in $0$: 

$\vec{y} = [\textbf{W}]\cdot \vec{x} + \vec{b}$.

In [51]:
dummy_input = tf.ones((2, 2))

In [52]:
dummy_input

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[1., 1.],
       [1., 1.]], dtype=float32)>

In [53]:
zero_layer_operation = ZeroLinearOperator(2, 2)

Layer has 4 total weights in a weight kernel of dimension 2x2


In [54]:
zero_layer_operation(dummy_input)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[0., 0.],
       [0., 0.]], dtype=float32)>

In [55]:
zero_layer_operation(tf.random.uniform(
    shape = (2, 1),
    minval = -1.,
    maxval = 1.0
))

InvalidArgumentError: Exception encountered when calling ZeroLinearOperator.call().

[1m{{function_node __wrapped____MklMatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [2,1], In[1]: [2,2] [Op:MatMul] name: [0m

Arguments received by ZeroLinearOperator.call():
  • inputs=tf.Tensor(shape=(2, 1), dtype=float32)

Wow. Recall that `matmul` is matrix-multiplication. I am not understanding the dimensions here. It seems that the input is a matrix and the weight kernel is a matrix, but the bias component is a VECTOR of dimensions $\text{rows} \times 1$.

In [ ]:
class OnesLinearOperator(keras.layers.Layer):
    
    def __init__(self, units = 5, input_dimension = 5):

        super().__init__() # inherit `Layer` properties and methods

        self.w = self.add_weight(
            shape = (input_dimension, units),
            initializer = 'ones',
            trainable = True
        )

        self.b = self.add_weight(
            shape = (units, ), 
            initializer = "zeros",
            trainable = True
        )

    def call(self, inputs):
        return tf.linalg.matvec(self.w, inputs) + self.b

In [ ]:
uniform_vector = tf.random.uniform(
    shape = (2, 1),
    minval = -1.0,
    maxval = 1.0
)

In [ ]:
ones_operator = OnesLinearOperator(2, 2)

In [ ]:
ones_operator(uniform_vector)

InvalidArgumentError: Exception encountered when calling OnesLinearOperator.call().

[1m{{function_node __wrapped____MklBatchMatMulV2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [2,2], In[1]: [2,1,1] 0 0 [Op:BatchMatMulV2] name: [0m

Arguments received by OnesLinearOperator.call():
  • inputs=tf.Tensor(shape=(2, 1), dtype=float32)

Do I not understand matrix-vector products?

In [ ]:
test_matrix = tf.constant(
    [[1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]]
)

test_vector = tf.constant(
    [1.0, -1.0, 0.0]
)

print(f"{test_matrix.shape[0]}x{test_matrix.shape[1]} matrix times {test_vector.shape}x1 vector = {tf.linalg.matvec(test_matrix, test_vector)}")

2x3 matrix times (3,)x1 vector = [-1. -1.]


I agree with that. I might be projecting what I think ought to happen in a layer computation onto the actual way the thing is coded up. That is evidenced by my equation that I wrote above; I am trying to coerce the code to do that for psychological comfort and mathematical clarity.

In [ ]:
random_vector = tf.random.uniform(
    shape = (2, 1)
)

In [ ]:
random_vector

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[0.949419  ],
       [0.44789565]], dtype=float32)>

It appears that the "hidden dimension" here is actually the batch size... That is... jank. That is nuts. So, it's the *rows* that represent the batches and the *columns* that represent the features????

In [ ]:
batch_size = 1
number_of_features = 5

test_vector_2 = tf.random.uniform(
    shape = (batch_size, number_of_features),
    minval = -5.0,
    maxval = 5.0
)

test_vector_2

<tf.Tensor: shape=(1, 5), dtype=float32, numpy=
array([[-1.8301606,  4.077488 ,  4.368374 , -1.1119962,  2.5416126]],
      dtype=float32)>

In [ ]:
OnesLinearOperator(5, number_of_features)(test_vector_2)

<tf.Tensor: shape=(1, 5), dtype=float32, numpy=array([[8.045318, 8.045318, 8.045318, 8.045318, 8.045318]], dtype=float32)>

Still don't get it. One more iteration might help...

In [ ]:
class FancyLayer1(keras.layers.Layer):
    
    def __init__(self, units = 5, input_dimension = 5):

        super().__init__() # inherit `Layer` properties and methods

        self.w = self.add_weight(
            shape = (input_dimension, units),
            initializer = 'random_normal',
            trainable = True
        )

        self.b = self.add_weight(
            shape = (units, ), 
            initializer = "random_normal",
            trainable = True
        )

    def call(self, inputs):
        return tf.linalg.matmul(inputs, self.w) + self.b

In [ ]:
new_layer_1 = FancyLayer1(10, 5)

The following SHOULD work if I understand this properly...

In [ ]:
test_vector_2

<tf.Tensor: shape=(1, 5), dtype=float32, numpy=
array([[-1.8301606,  4.077488 ,  4.368374 , -1.1119962,  2.5416126]],
      dtype=float32)>

In [ ]:
new_layer_1(test_vector_2)

<tf.Tensor: shape=(1, 10), dtype=float32, numpy=
array([[ 0.55948   ,  0.5230461 ,  0.08069988,  0.0232573 , -0.21279934,
        -0.21767569,  0.39285544,  0.08754447,  0.05898876,  0.53371143]],
      dtype=float32)>

It turned out that the number of `units` is the final dimension after the matrix-matrix multiplication. It seems like what is happening is: $\vec{v}^{T} \cdot [\textbf{W}]$.

In [ ]:
new_layer_1.weights

[<Variable path=fancy_layer1_1/variable_18, shape=(5, 10), dtype=float32, value=[[ 0.00840616 -0.01468539  0.01631398 -0.09530421  0.02576505  0.07586604
   -0.05831044  0.00502612 -0.04409659 -0.01051636]
  [ 0.10097289  0.04844328  0.04518101 -0.05456001 -0.08326738 -0.03011723
    0.08129628 -0.09146627  0.00363488  0.06143916]
  [ 0.00827181  0.08068048 -0.00475063  0.01377023  0.04560642 -0.02811048
   -0.02998784  0.10540064 -0.00570616  0.08617053]
  [-0.06730896 -0.00703603 -0.04521165  0.0571164   0.06620594 -0.02545435
    0.02305802 -0.02388613  0.05278337  0.03805856]
  [ 0.02429479 -0.03466836 -0.01625053  0.00802386  0.02900897  0.01825499
    0.06369967 -0.00518693  0.03138311 -0.0101522 ]]>,
 <Variable path=fancy_layer1_1/variable_19, shape=(10,), dtype=float32, value=[-0.00958072  0.02648948 -0.06188805  0.05426924 -0.02545809  0.09206879
  -0.05060802 -0.00411183 -0.03267846 -0.04435378]>]

In [ ]:
new_layer_1.w

<Variable path=fancy_layer1_1/variable_18, shape=(5, 10), dtype=float32, value=[[ 0.00840616 -0.01468539  0.01631398 -0.09530421  0.02576505  0.07586604
  -0.05831044  0.00502612 -0.04409659 -0.01051636]
 [ 0.10097289  0.04844328  0.04518101 -0.05456001 -0.08326738 -0.03011723
   0.08129628 -0.09146627  0.00363488  0.06143916]
 [ 0.00827181  0.08068048 -0.00475063  0.01377023  0.04560642 -0.02811048
  -0.02998784  0.10540064 -0.00570616  0.08617053]
 [-0.06730896 -0.00703603 -0.04521165  0.0571164   0.06620594 -0.02545435
   0.02305802 -0.02388613  0.05278337  0.03805856]
 [ 0.02429479 -0.03466836 -0.01625053  0.00802386  0.02900897  0.01825499
   0.06369967 -0.00518693  0.03138311 -0.0101522 ]]>

In [ ]:
new_layer_1.w.shape

TensorShape([5, 10])

In [ ]:
new_layer_1.b

<Variable path=fancy_layer1_1/variable_19, shape=(10,), dtype=float32, value=[-0.00958072  0.02648948 -0.06188805  0.05426924 -0.02545809  0.09206879
 -0.05060802 -0.00411183 -0.03267846 -0.04435378]>

## (2): Nontrainable Weights:

Now, we need to make a basic thing that adds stuff with nontrainable weights. Alright.

In [ ]:
class SumLayer(keras.layers.Layer):
    def __init__(self, input_dimension: int = 2):
        super().__init__()
        
        self.total = self.add_weight(
            initializer = "zeros",
            shape = (input_dimension, ),
            trainable = False
        )

    def call(self, input_values):
        self.total.assign_add(tf.reduce_sum(input_values, axis = 0))
        return self.total

In [ ]:
some_matrix = tf.ones(shape = (2, 2))
some_matrix

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[1., 1.],
       [1., 1.]], dtype=float32)>

In [57]:
SumLayer(2)(some_matrix)

<Variable path=sum_layer_3/variable_25, shape=(2,), dtype=float32, value=[2. 2.]>

In [58]:
SumLayer(2)(some_matrix)

<Variable path=sum_layer_4/variable_26, shape=(2,), dtype=float32, value=[2. 2.]>

In [59]:
SumLayer(2)(some_matrix)

<Variable path=sum_layer_5/variable_27, shape=(2,), dtype=float32, value=[2. 2.]>

Um.. I am not correctly getting what they are getting on this page... 

In [62]:
x = tf.ones((2, 2))
x

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[1., 1.],
       [1., 1.]], dtype=float32)>

In [63]:
my_sum = SumLayer(2)

In [64]:
my_sum(x)

<Variable path=sum_layer_6/variable_28, shape=(2,), dtype=float32, value=[2. 2.]>

In [65]:
my_sum(x)

<Variable path=sum_layer_6/variable_28, shape=(2,), dtype=float32, value=[4. 4.]>

In [66]:
my_sum(x)

<Variable path=sum_layer_6/variable_28, shape=(2,), dtype=float32, value=[6. 6.]>

Alright... NOW it's "saving the state" in the nontrainable weights...

In [67]:
my_sum.weights

[<Variable path=sum_layer_6/variable_28, shape=(2,), dtype=float32, value=[6. 6.]>]

List of TF Variables.

In [69]:
my_sum.non_trainable_weights

[<Variable path=sum_layer_6/variable_28, shape=(2,), dtype=float32, value=[6. 6.]>]

In [70]:
my_sum.trainable_weights

[]

## (3): Building Weights

In [81]:
class BuildOnTime(keras.layers.Layer):
    def __init__(self, number_of_nodes: int = 32):
        super().__init__()
        self.number_of_nodes = number_of_nodes
    
    def build(self, input_shape):
        self.w = self.add_weight(
            shape = (input_shape[-1], self.number_of_nodes),
            initializer = "random_normal",
            trainable = True,
        )
        
        self.b = self.add_weight(
            shape = (self.number_of_nodes, ),
            initializer = "random_normal",
            trainable = True
        )

    def call(self, layer_inputs):
        return tf.matmul(layer_inputs, self.w) + self.b

This is key: `(input_shape[-1], self.number_of_nodes)`. The number of "rows" in the weight kernel is equal to the number of COLUMNS in the input data... Uh...

In [82]:
cool_layer = BuildOnTime(30) # should have 30 nodes

In [83]:
x

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[1., 1.],
       [1., 1.]], dtype=float32)>

In [84]:
cool_layer(x)

<tf.Tensor: shape=(2, 30), dtype=float32, numpy=
array([[-0.13879332,  0.02145722, -0.09685569,  0.03537625, -0.07800563,
         0.02313061, -0.00670588,  0.06066458, -0.05880656,  0.02528771,
        -0.00732492, -0.01062139, -0.21145494, -0.06774747, -0.01069463,
        -0.03637953, -0.00832175,  0.1476204 ,  0.05220336,  0.07598498,
        -0.15476993,  0.01607896, -0.11424454, -0.05824447,  0.08161175,
        -0.07959504,  0.03664825, -0.01738067,  0.08640997,  0.00087401],
       [-0.13879332,  0.02145722, -0.09685569,  0.03537625, -0.07800563,
         0.02313061, -0.00670588,  0.06066458, -0.05880656,  0.02528771,
        -0.00732492, -0.01062139, -0.21145494, -0.06774747, -0.01069463,
        -0.03637953, -0.00832175,  0.1476204 ,  0.05220336,  0.07598498,
        -0.15476993,  0.01607896, -0.11424454, -0.05824447,  0.08161175,
        -0.07959504,  0.03664825, -0.01738067,  0.08640997,  0.00087401]],
      dtype=float32)>

I still don't get it at all.

## (4): Recursive Composition

In [94]:
class MLPBlock(keras.layers.Layer):
    def __init__(self):
        super().__init__()

        self.layer_1 = BuildOnTime(10)
        self.layer_2 = BuildOnTime(10)
        self.layer_3 = BuildOnTime(10)
        self.layer_4 = BuildOnTime(10)
        self.model_output = BuildOnTime(1)

        # guess: 4x10 + 4x(10x10) = 440 total parameters
        # that will be WRONG because I don't get the shapes here...

    def call(self, model_inputs):
        x = self.layer_1(model_inputs)
        x = tf.nn.relu(x)
        x = self.layer_2(x)
        x = tf.nn.relu(x)
        x = self.layer_3(x)
        x = tf.nn.relu(x)
        x = self.layer_4(x)
        x = tf.nn.relu(x)
        x = self.model_output(x)
        return x

In [95]:
mlp4 = MLPBlock()

In [96]:
mlp4(tf.ones(
    shape = (3, 64) # 3 rows, 64 columns
    ))

<tf.Tensor: shape=(3, 1), dtype=float32, numpy=
array([[0.04831095],
       [0.04831095],
       [0.04831095]], dtype=float32)>

In [98]:
len(mlp4.weights)

10

What... Is this the total number of layers?

In [100]:
mlp4.weights # all weights + biases 

[<Variable path=mlp_block_3/build_on_time_18/variable_43, shape=(64, 10), dtype=float32, value=[[ 4.57366109e-02 -1.78814307e-02 -2.10099909e-02  2.14206781e-02
   -1.11327626e-01  6.59376383e-02  1.77925434e-02  7.52953216e-02
   -2.99611241e-02 -8.58086050e-02]
  [ 9.09711868e-02  3.81576741e-04 -1.71024781e-02  2.73257308e-03
    4.12307568e-02  3.43666561e-02  4.61976044e-02 -3.04331016e-02
   -3.04776039e-02 -1.29622510e-02]
  [ 5.91792874e-02 -8.09189156e-02 -1.75876301e-02 -1.26194775e-01
   -7.93349966e-02 -3.36897373e-02 -2.15227492e-02 -1.45426141e-02
    2.64983419e-02  1.88576116e-03]
  [-8.35137814e-02  1.35627659e-02  5.60178421e-02  1.02049492e-01
    6.18579388e-02 -3.21539119e-02  1.85568314e-02  9.07931477e-02
    8.08730274e-02  4.05022912e-02]
  [ 6.28993213e-02 -4.15419303e-02  6.08632267e-02  2.09486764e-02
    5.92913516e-02  4.55898158e-02  3.12569812e-02  3.00825736e-03
   -1.49114147e-01 -2.62498092e-02]
  [-1.21942665e-02 -5.44745326e-02  1.19586280e-02 -1.64

In [102]:
len(mlp4.trainable_weights)

10

## (5): Add Loss

In [104]:
class Regularization(keras.layers.Layer):
    def __init__(self, rate = 1e-2):
        super().__init__()
        self.rate = rate

    def call(self, model_inputs):
        self.add_loss(self.rate * tf.reduce_mean(model_inputs))
        return model_inputs